# 📈 Notebook 5 — Evaluation & Metrics
**Pill Counter | Computer Vision Pipeline**

> **Goal:** Rigorous evaluation of the trained model — mAP@0.5, mAP@0.5:0.95, precision-recall curves, confusion matrix, and per-image qualitative inspection.

---
**Key outputs:**
- `results/metrics/evaluation_report.json`
- Precision-recall curve plot
- Confusion matrix plot
- Qualitative detection grid


## 5.1 — Imports & Load Model

In [ ]:
import torch, yaml, json, cv2, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

with open("config.yaml") as f:
    CFG = yaml.safe_load(f)

MODEL_PATH   = Path(CFG["paths"]["trained_models"]) / "best.pt"
DATASET_YAML = Path(CFG["paths"]["processed_data"]) / "dataset.yaml"
DEVICE       = "cuda:0" if torch.cuda.is_available() else "cpu"
CONF         = CFG["inference"]["confidence_threshold"]
IOU          = CFG["inference"]["nms_threshold"]

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH} — run Notebook 4 first."
model = YOLO(str(MODEL_PATH))
print(f"✅  Model loaded: {MODEL_PATH}")
print(f"   Device: {DEVICE}  Conf: {CONF}  IoU: {IOU}")


## 5.2 — Validation Set Metrics

In [ ]:
print("Running validation...")
val_metrics = model.val(
    data=str(DATASET_YAML),
    split="val",
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    verbose=True,
)

# Extract key scalar metrics
results_dict = {
    "mAP50"       : float(val_metrics.box.map50),
    "mAP50_95"    : float(val_metrics.box.map),
    "precision"   : float(val_metrics.box.mp),
    "recall"      : float(val_metrics.box.mr),
}

print("\n── Validation Results ──────────────────")
for k, v in results_dict.items():
    print(f"  {k:15s}: {v:.4f}")


## 5.3 — Test Set Metrics

In [ ]:
print("Running test set evaluation...")
test_metrics = model.val(
    data=str(DATASET_YAML),
    split="test",
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    verbose=False,
)

test_results = {
    "mAP50"    : float(test_metrics.box.map50),
    "mAP50_95" : float(test_metrics.box.map),
    "precision": float(test_metrics.box.mp),
    "recall"   : float(test_metrics.box.mr),
}

print("── Test Results ────────────────────────")
for k, v in test_results.items():
    print(f"  {k:15s}: {v:.4f}")

# Combined bar chart
fig, ax = plt.subplots(figsize=(9, 4))
metrics = list(results_dict.keys())
val_vals  = list(results_dict.values())
test_vals = list(test_results.values())
x = np.arange(len(metrics))
w = 0.35
ax.bar(x - w/2, val_vals,  w, label="Val",  color="#4C8BF5")
ax.bar(x + w/2, test_vals, w, label="Test", color="#F5844C")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Validation vs Test Metrics"); ax.legend()
for i, (vv, tv) in enumerate(zip(val_vals, test_vals)):
    ax.text(i-w/2, vv+0.01, f"{vv:.3f}", ha="center", fontsize=8)
    ax.text(i+w/2, tv+0.01, f"{tv:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("results/visualizations/val_vs_test_metrics.png", bbox_inches="tight")
plt.show()


## 5.4 — Confidence Threshold Sweep

> Find the optimal confidence threshold that maximises F1 score on the validation set.

In [ ]:
conf_thresholds = np.arange(0.1, 0.95, 0.05)
f1_scores, precisions, recalls = [], [], []

for conf in conf_thresholds:
    m = model.val(data=str(DATASET_YAML), split="val",
                  device=DEVICE, conf=float(conf), iou=IOU, verbose=False)
    p = float(m.box.mp); r = float(m.box.mr)
    f1 = 2*p*r/(p+r+1e-9)
    precisions.append(p); recalls.append(r); f1_scores.append(f1)

best_idx  = int(np.argmax(f1_scores))
best_conf = conf_thresholds[best_idx]
best_f1   = f1_scores[best_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(conf_thresholds, f1_scores,  label="F1",       color="#4C8BF5", linewidth=2)
ax.plot(conf_thresholds, precisions, label="Precision", color="#F5844C", linewidth=1.5, linestyle="--")
ax.plot(conf_thresholds, recalls,    label="Recall",    color="#4CF584", linewidth=1.5, linestyle="--")
ax.axvline(best_conf, color="red", linestyle=":", linewidth=1.5, label=f"Best conf={best_conf:.2f}")
ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Confidence Threshold Sweep — F1/Precision/Recall"); ax.legend()
plt.tight_layout()
plt.savefig("results/visualizations/conf_threshold_sweep.png", bbox_inches="tight")
plt.show()

print(f"✅  Optimal confidence threshold: {best_conf:.2f}  (F1 = {best_f1:.4f})")
print(f"   Update config.yaml inference.confidence_threshold → {best_conf:.2f}")


## 5.5 — Qualitative Detection Grid

In [ ]:
with open(DATASET_YAML) as f:
    ds = yaml.safe_load(f)

test_img_dir = Path(ds["path"]) / ds["test"]
test_images  = list(test_img_dir.glob("*.*"))[:16]

n_show = min(8, len(test_images))
random.shuffle(test_images)
sample_imgs = test_images[:n_show]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Model Predictions on Test Images", fontsize=13, fontweight="bold")
axes = axes.flatten()

for ax_i, img_path in enumerate(sample_imgs):
    img = cv2.imread(str(img_path))
    if img is None: continue

    preds = model(img, conf=best_conf, iou=IOU, device=DEVICE, verbose=False)[0]
    drawn = img.copy()
    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf_v = float(box.conf[0])
        cv2.rectangle(drawn, (x1, y1), (x2, y2), (0, 200, 80), 2)
        cv2.putText(drawn, f"pill {conf_v:.2f}", (x1, max(y1-5,10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 80), 1)

    axes[ax_i].imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    axes[ax_i].set_title(f"{len(preds.boxes)} detection(s)", fontsize=9)
    axes[ax_i].axis("off")

for ax in axes[n_show:]: ax.axis("off")
plt.tight_layout()
plt.savefig("results/visualizations/qualitative_detections.png", bbox_inches="tight")
plt.show()


## 5.6 — Save Evaluation Report

In [ ]:
report = {
    "model_path": str(MODEL_PATH),
    "device": DEVICE,
    "optimal_confidence": float(best_conf),
    "optimal_f1": float(best_f1),
    "val_metrics": results_dict,
    "test_metrics": test_results,
}

report_path = Path("results/metrics/evaluation_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"✅  Evaluation report saved: {report_path}")
print(json.dumps(report, indent=2))


## ✅  Notebook 5 Complete
> Evaluation done. Proceed to **Notebook 6 — Real-Time Inference**.